# Setup

In [ ]:
# Optional autoreload while developing locally

%load_ext autoreload
%autoreload 2

In [ ]:

# 0) Local-only setup and base diagnostics
import os
import sys
import platform
from pathlib import Path

print("Python:", sys.version)
print("Platform:", platform.platform())

PROJECT_DIR = Path.cwd()



In [ ]:

# 0.3) Shared device diagnostics
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())
    print("GPU 0:", torch.cuda.get_device_name(0))
    DEVICE = "cuda:0"
else:
    DEVICE = "cpu"
print("Using DEVICE =", DEVICE)


# Imports

In [ ]:
import pickle
import re

import pandas as pd
from spacy.tokens import Doc


from tokenizer import create_tokenizer, ensure_booknlp_extensions, tokens_to_dicts
from doc_chunker import create_chunker
from runtime_config import (
    RUNTIME_PROFILE,
    DEVICE,
    CHUNK_SIZE,
    OVERLAP_SENTENCES,
    MAX_EXPANDED_CHUNK_TOKENS,
)

In [ ]:
from coreference.coreference_sub_orchestrator import (
    run_coreference_resolution,
    print_non_singleton_coref_clusters,
)

from coreference.coref_schema import require_coref_layer

In [ ]:
from ocean.ocean_probability_scoring import (
    ContextConfig,
    MentionRenderingConfig,
    OceanScoringConfig,
    OceanWeightConfig,
    DirectNLIConfig,
    OceanProbabilityExportConfig,
    export_ocean_probability_csvs,
)

from ocean.ocean_annotator import annotate_doc_with_ocean_folder
from ocean.ocean_schema import require_ocean_layer


# Functions

In [ ]:
# text loading + preprocessing
def load_txt(path):
    path = Path(path)

    try:
        return path.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        return path.read_text(encoding="latin-1")

In [ ]:
def preprocess_for_NER(text):
    # Normalize Windows/Mac newlines
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Remove possible invisible BOM
    text = text.replace("\ufeff", "")

    # Normalize curly apostrophes/quotes
    text = text.replace("’", "'")
    text = text.replace("‘", "'")
    text = text.replace("“", '"').replace("”", '"')

    # Replace line breaks inside paragraphs with spaces,
    # but preserve paragraph boundaries
    paragraphs = text.split("\n\n")
    paragraphs = [
        re.sub(r"\s*\n\s*", " ", paragraph).strip()
        for paragraph in paragraphs
        if paragraph.strip()
    ]

    text = "\n\n".join(paragraphs)

    chapter_re = re.compile(
        r"(?im)^\s*chapter\s+(?:[ivxlcdm]+|\d+)\b[^\n]*\n*"
    )

    text = chapter_re.sub("", text)

    # Normalize multiple spaces
    text = re.sub(r"[ \t]+", " ", text)

    return text.strip()

In [ ]:

def resolve_text_path(
    *,
    project_dir: Path,
    filename: str = "oz_full.txt",
    local_path: str | Path = "../../datasets/libri/oz_full.txt",
) -> Path:
    """Resolve the local input text path."""
    text_path = Path(local_path)
    if text_path.exists():
        return text_path

    fallback_path = project_dir / filename
    if fallback_path.exists():
        return fallback_path

    raise FileNotFoundError(
        "Could not resolve a local text file. "
        f"Tried LOCAL_TEXT_PATH={text_path} and PROJECT_DIR / {filename!r} = {fallback_path}."
    )


In [ ]:
def save_doc(doc, path: str | Path) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("wb") as f:
        pickle.dump(doc, f)
    return path


def load_doc(path: str | Path):
    # BookNLP/tokenizer extensions must be registered before unpickling the Doc.
    # The global coreference extension is registered inside the coreference
    # sub-orchestrator immediately before final annotation.
    ensure_booknlp_extensions()
    path = Path(path)
    with path.open("rb") as f:
        return pickle.load(f)


# Config

## I/O config

In [ ]:
# Requested local file
TEXT_FILENAME = "oz_full.txt"
LOCAL_TEXT_PATH = Path(f"../../datasets/libri/{TEXT_FILENAME}")

TEXT_PATH = resolve_text_path(
    project_dir=PROJECT_DIR,
    filename=TEXT_FILENAME,
    local_path=LOCAL_TEXT_PATH,
)

OUTPUT_ROOT = PROJECT_DIR / "outputs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

TOKENIZED_DOC_PATH = OUTPUT_ROOT / "tokenized_doc.pkl"
ANNOTATED_DOC_PATH = OUTPUT_ROOT / "annotated_doc.pkl"
OCEAN_SCORED_DOC_PATH = OUTPUT_ROOT / "ocean_scored_doc.pkl"

print("TEXT_PATH =", TEXT_PATH)
print("OUTPUT_ROOT =", OUTPUT_ROOT)
print("TOKENIZED_DOC_PATH =", TOKENIZED_DOC_PATH)
print("ANNOTATED_DOC_PATH =", ANNOTATED_DOC_PATH)


In [ ]:
GLOBAL_COREF_OUTPUT_DIR = OUTPUT_ROOT / "global_coref"
GLOBAL_COREF_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("GLOBAL_COREF_OUTPUT_DIR =", GLOBAL_COREF_OUTPUT_DIR)

## Runtime and pipeline config

In [ ]:
# Pipeline configuration
SPACY_MODEL = "en_core_web_sm"
TOKENIZER_DISABLE = ("ner",)

print("RUNTIME_PROFILE =", RUNTIME_PROFILE)
print("DEVICE =", DEVICE)
print("CHUNK_SIZE(expanded) =", CHUNK_SIZE)
print("OVERLAP_SENTENCES =", OVERLAP_SENTENCES)
print("MAX_EXPANDED_CHUNK_TOKENS =", MAX_EXPANDED_CHUNK_TOKENS)
print("GLOBAL_COREF_OUTPUT_DIR =", GLOBAL_COREF_OUTPUT_DIR)


# Main

### Tokenization

In [ ]:
# Load or create the stable tokenized Doc artifact.
if TOKENIZED_DOC_PATH.exists():
    print(f"[doc] Loading tokenized Doc from {TOKENIZED_DOC_PATH}")
    doc = load_doc(TOKENIZED_DOC_PATH)
else:
    print("[doc] Tokenized Doc not found; creating it from raw text.")
    raw_text = load_txt(TEXT_PATH)
    text = preprocess_for_NER(raw_text)
    tokenizer = create_tokenizer(
        SPACY_MODEL,
        disable=TOKENIZER_DISABLE,
        verbose=True,
    )
    doc = tokenizer.tokenize(text)
    save_doc(doc, TOKENIZED_DOC_PATH)
    print(f"[doc] Saved tokenized Doc to {TOKENIZED_DOC_PATH}")

print("Doc tokens:", len(doc))
print("Doc sentences:", sum(1 for _ in doc.sents))

### Chunking

In [ ]:
chunker = create_chunker(
    chunk_size=CHUNK_SIZE,
    overlap_sentences=OVERLAP_SENTENCES,
    max_expanded_chunk_tokens=MAX_EXPANDED_CHUNK_TOKENS,
)
chunk_plan = chunker.plan(doc)

### Coreference clusters extraction

In [ ]:
if ANNOTATED_DOC_PATH.exists():
    print(f"[coref-annotation] Loading coref-annotated Doc from {ANNOTATED_DOC_PATH}")
    doc = load_doc(ANNOTATED_DOC_PATH)
else:
    print("[coref-annotation] Annotated Doc not found; calling annotator sub-orchestrator.")
    doc = run_coreference_resolution(
        doc=doc,
        chunker=chunker,
        chunk_plan=chunk_plan,
        document_id=TEXT_PATH.stem,
        output_dir=GLOBAL_COREF_OUTPUT_DIR,
    )
    save_doc(doc, ANNOTATED_DOC_PATH)
    print("[coref-annotation] Saved annotated Doc to:", ANNOTATED_DOC_PATH)
    
    
if not Doc.has_extension("coref_layer"):
    Doc.set_extension("coref_layer", default=None)

In [ ]:
print("[coref-annotation] Summary:")
print(doc._.coref_layer.summary())
print_non_singleton_coref_clusters(doc)

### OCEAN scoring

In [ ]:
cluster_ids = [8, 97, 153, 48, 111, 76, 226, 20, 41, 4, 252, 67, 35, 540, 238, 335, 292, 201, 460, 533, 326, 36, 2, 22, 221, 387]
n_mentions = None  # None = score ALL mentions for each requested cluster
random_seed = 67

In [ ]:
if OCEAN_SCORED_DOC_PATH.exists():
    print(f"[OCEAN scoring] Loading ocean_scoring-annotated Doc from {OCEAN_SCORED_DOC_PATH}")
    doc = load_doc(OCEAN_SCORED_DOC_PATH)
else:

    csv_paths_by_cluster_id = export_ocean_probability_csvs(
    doc,
    OceanProbabilityExportConfig(
        cluster_ids=cluster_ids,
        n_mentions_per_cluster=n_mentions,
        output_root="./outputs",

        context_config=ContextConfig(
            n_sentences_before=0,
            n_sentences_after=0,
            mark_mention=True,
            deduplicate=False,
        ),
        rendering_config=MentionRenderingConfig(
            canonicalize_simple_mentions=True,
            keep_first_second_person=True,
        ),
        scoring_config=OceanScoringConfig(
            subject_aware=True,
        ),
        weight_config=OceanWeightConfig(),
        nli_config=DirectNLIConfig(
            pair_batch_size=64, # on stronger hardware can be safely set to 64
        ),

        chunk_size=64,

        overwrite_csv=False,
        resume_from_csv=True,
        use_sqlite_cache=True,

        random_seed=random_seed,
        sort_sample_by_cluster_order=True,
        return_dataframes=False,
        print_progress=True,
    ),
)


    print(csv_paths_by_cluster_id)


    n_part = "all" if n_mentions is None else str(n_mentions)
    ocean_folder = Path("./outputs") / "OCEAN_profiles" / n_part

    doc = annotate_doc_with_ocean_folder(
        doc,
        ocean_folder,
    )
    save_doc(doc, OCEAN_SCORED_DOC_PATH)
    print("\n\n\n[OCEAN scoring] Saved annotated Doc to:", OCEAN_SCORED_DOC_PATH)

ocean = require_ocean_layer(doc)
print(ocean.summary())
#### test

In [ ]:
coref = doc._.coref_layer
ocean = doc._.ocean_layer

for cluster_id in cluster_ids:
    print(coref.clusters[cluster_id].canonical_name)
    print(ocean.cluster(cluster_id).scores.as_dict())